Detección y Segmentación con YOLO (Transfer Learning)_COCO

**Instructor:** Dr Mario Iván López Valdovinos  
**Alumno:** Cesar Eduardo Inda Ceniceros  



    Es proyecto implementa y evalúa dos modelos de red neuronal (YOLOv8) mediante **transfer learning** a partir de pesos preentrenados en COCO, para las tareas de:

    1. **Detección de objetos** (bounding boxes)
    2. **Segmentación de instancias** (máscaras a nivel de píxel)

    Se evalúa el desempeño con las métricas estándar: Precision, Recall, F1-score, IoU, Dice y mAP (en los umbrales 0.50 y 0.50:0.95).

    **Dataset:** se utiliza `coco128-seg`, un subconjunto oficial de 128 imágenes tomadas de COCO train2017, que ya incluye anotaciones de segmentación (polígonos) además de bounding boxes. Este subconjunto es distribuido por Ultralytics específicamente para validar pipelines de entrenamiento/transfer learning sin necesitar los ~20 GB del dataset COCO completo.

    > Nota metodológica: dado que el ejercicio pide explícitamente *transfer learning* (no entrenamiento desde cero), partimos siempre de los pesos `yolov8s.pt` / `yolov8s-seg.pt` preentrenados en el COCO completo (80 clases), y hacemos fine-tuning sobre nuestra partición.


## 1. Instalación de librerías

Instalamos PyTorch (con soporte CUDA), Ultralytics YOLO, y las utilerías necesarias para manejo de datos, métricas y visualización.

In [1]:
# Si ya tienes PyTorch con CUDA instalado (recomendado, para no reinstalar torch),
# comenta la línea de torch y deja solo el resto.
!pip install ultralytics matplotlib pycocotools scikit-learn seaborn opencv-python --quiet

import ultralytics
ultralytics.checks()  # imprime versión, disponibilidad de CUDA, GPU detectada, etc.


Ultralytics 8.4.124  Python-3.11.13 torch-2.12.1+cpu CPU (Intel Core i7-9750H 2.60GHz)
Setup complete  (12 CPUs, 15.9 GB RAM, 432.7/465.1 GB disk)


In [2]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
DEVICE = 0 if torch.cuda.is_available() else "cpu"


PyTorch version: 2.12.1+cpu
CUDA disponible: False


## 2. Descarga del dataset COCO (subconjunto)

Usamos `coco128-seg.zip`, distribuido oficialmente por Ultralytics. Ya viene en formato YOLO (imágenes + labels `.txt` con polígonos normalizados), lo cual nos ahorra el paso de convertir las anotaciones originales de COCO (formato JSON estilo COCO API) a formato YOLO.

Estructura que se descarga:
```
coco128-seg/
├── images/train2017/   (128 imágenes)
└── labels/train2017/   (128 archivos .txt, uno por imagen, con polígonos de segmentación)
```

Como el dataset original solo trae una partición (`train2017`), en la sección 3 lo re-particionamos nosotros mismos en 70/15/15 como pide el laboratorio.


In [3]:
import os
from pathlib import Path
from ultralytics.utils.downloads import download

DATA_ROOT = Path("./dataset_coco")
DATA_ROOT.mkdir(exist_ok=True)

# Descarga oficial del subconjunto (imágenes + labels de segmentación en formato YOLO)
url = "https://github.com/ultralytics/assets/releases/download/v0.0.0/coco128-seg.zip"
download(url, dir=DATA_ROOT, unzip=True, delete=True)

IMAGES_DIR = DATA_ROOT / "coco128-seg" / "images" / "train2017"
LABELS_DIR = DATA_ROOT / "coco128-seg" / "labels" / "train2017"

print(f"Imágenes descargadas: {len(list(IMAGES_DIR.glob('*.jpg')))}")
print(f"Labels descargados:   {len(list(LABELS_DIR.glob('*.txt')))}")


WARNING Skipping dataset_coco\coco128-seg.zip unzip as destination directory C:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\dataset_coco\coco128-seg is not empty.
Imágenes descargadas: 128
Labels descargados:   128


In [4]:
# Nombres de las 80 clases de COCO (mismo orden que usan los pesos preentrenados de YOLO)
COCO_CLASSES = ultralytics.YOLO("yolov8s.pt").names  # dict {id: nombre}
print(f"Número de clases COCO: {len(COCO_CLASSES)}")
list(COCO_CLASSES.items())[:10]


Número de clases COCO: 80


[(0, 'person'),
 (1, 'bicycle'),
 (2, 'car'),
 (3, 'motorcycle'),
 (4, 'airplane'),
 (5, 'bus'),
 (6, 'train'),
 (7, 'truck'),
 (8, 'boat'),
 (9, 'traffic light')]

## 3. Partición del dataset (70% train / 15% val / 15% test)

Reorganizamos las 128 imágenes descargadas en tres particiones independientes, manteniendo la correspondencia imagen–etiqueta. Usamos una semilla fija para que la partición sea reproducible.


In [5]:
import random
import shutil

random.seed(42)

SPLIT_ROOT = Path("./dataset_coco/split")
if SPLIT_ROOT.exists():
    shutil.rmtree(SPLIT_ROOT)

all_images = sorted(IMAGES_DIR.glob("*.jpg"))
random.shuffle(all_images)

n_total = len(all_images)
n_train = int(n_total * 0.70)
n_val = int(n_total * 0.15)
# el resto (15% o el residuo por redondeo) va a test
splits = {
    "train": all_images[:n_train],
    "val": all_images[n_train:n_train + n_val],
    "test": all_images[n_train + n_val:],
}

for split_name, img_list in splits.items():
    img_out = SPLIT_ROOT / "images" / split_name
    lbl_out = SPLIT_ROOT / "labels" / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    for img_path in img_list:
        label_path = LABELS_DIR / (img_path.stem + ".txt")
        shutil.copy(img_path, img_out / img_path.name)
        if label_path.exists():
            shutil.copy(label_path, lbl_out / label_path.name)

print(f"Total de imágenes: {n_total}")
for split_name, img_list in splits.items():
    pct = len(img_list) / n_total * 100
    print(f"  {split_name:5s}: {len(img_list):4d} imágenes ({pct:.1f}%)")


Total de imágenes: 128
  train:   89 imágenes (69.5%)
  val  :   19 imágenes (14.8%)
  test :   20 imágenes (15.6%)


## 4. Preparación de las anotaciones

Ultralytics YOLO usa un único formato de archivo de texto por imagen, pero el **contenido** de cada línea cambia según la tarea:

**Detección** (bounding box) — cada línea:
```
<clase> <x_centro> <y_centro> <ancho> <alto>
```
Todas las coordenadas normalizadas entre 0 y 1 respecto al tamaño de la imagen.

**Segmentación** (máscara) — cada línea:
```
<clase> <x1> <y1> <x2> <y2> ... <xn> <yn>
```
Es decir, el polígono completo del contorno del objeto, también normalizado.

`coco128-seg` ya trae las anotaciones en formato de **polígono** (segmentación). Para la tarea de **detección** derivamos el bounding box de cada objeto calculando el rectángulo delimitador (mínimo y máximo en x, y) de su polígono — así generamos las etiquetas de detección a partir de las de segmentación, garantizando que ambas tareas usen exactamente los mismos objetos.


In [6]:
def polygon_to_bbox(coords):
    """Convierte una lista plana [x1,y1,x2,y2,...] normalizada a un bbox
    YOLO [x_centro, y_centro, ancho, alto], también normalizado."""
    xs = coords[0::2]
    ys = coords[1::2]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)
    x_center = (x_min + x_max) / 2
    y_center = (y_min + y_max) / 2
    width = x_max - x_min
    height = y_max - y_min
    return x_center, y_center, width, height


def build_detection_labels(seg_labels_dir: Path, det_labels_dir: Path):
    det_labels_dir.mkdir(parents=True, exist_ok=True)
    for seg_file in seg_labels_dir.glob("*.txt"):
        lines_out = []
        with open(seg_file) as f:
            for line in f:
                parts = line.strip().split()
                if not parts:
                    continue
                cls = parts[0]
                coords = list(map(float, parts[1:]))
                xc, yc, w, h = polygon_to_bbox(coords)
                lines_out.append(f"{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")
        with open(det_labels_dir / seg_file.name, "w") as f:
            f.write("\n".join(lines_out))


# Generamos labels de detección (bbox) a partir de los labels de segmentación,
# para cada partición train/val/test
for split_name in ["train", "val", "test"]:
    seg_dir = SPLIT_ROOT / "labels" / split_name
    det_dir = SPLIT_ROOT / "labels_det" / split_name
    build_detection_labels(seg_dir, det_dir)
    print(f"Labels de detección generados para '{split_name}': "
          f"{len(list(det_dir.glob('*.txt')))} archivos")


Labels de detección generados para 'train': 87 archivos
Labels de detección generados para 'val': 19 archivos
Labels de detección generados para 'test': 20 archivos


## 5. Archivos de configuración del dataset

YOLO necesita un archivo `.yaml` que indique dónde están las imágenes de cada partición y el listado de clases. Creamos **dos** archivos: uno para detección (apunta a `labels_det/`) y otro para segmentación (apunta a `labels/`, que ya contiene los polígonos).


In [7]:
import yaml

names_dict = {i: name for i, name in COCO_CLASSES.items()}

def write_yaml(path, labels_subdir):
    cfg = {
        "path": str(SPLIT_ROOT.resolve()),
        "train": f"images/train",
        "val": f"images/val",
        "test": f"images/test",
        "names": names_dict,
    }
    with open(path, "w") as f:
        yaml.dump(cfg, f, sort_keys=False, allow_unicode=True)

# NOTA: Ultralytics infiere la carpeta de labels reemplazando "images" por "labels"
# en la ruta. Como tenemos dos variantes de labels (labels/ y labels_det/), creamos
# una copia de la estructura de imágenes apuntando a la carpeta de labels correcta
# mediante symlinks, para no duplicar las imágenes en disco.

def make_task_root(task_labels_dirname, task_root_name):
    task_root = Path(f"./dataset_coco/{task_root_name}")
    for split_name in ["train", "val", "test"]:
        img_src = (SPLIT_ROOT / "images" / split_name).resolve()
        lbl_src = (SPLIT_ROOT / task_labels_dirname / split_name).resolve()
        img_dst = task_root / "images" / split_name
        lbl_dst = task_root / "labels" / split_name
        img_dst.parent.mkdir(parents=True, exist_ok=True)
        lbl_dst.parent.mkdir(parents=True, exist_ok=True)
        if img_dst.exists() or img_dst.is_symlink():
            img_dst.unlink()
        if lbl_dst.exists() or lbl_dst.is_symlink():
            lbl_dst.unlink()
        img_dst.symlink_to(img_src)
        lbl_dst.symlink_to(lbl_src)
    return task_root

DET_ROOT = make_task_root("labels_det", "det_task")
SEG_ROOT = make_task_root("labels", "seg_task")

def write_task_yaml(path, task_root):
    cfg = {
        "path": str(task_root.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": names_dict,
    }
    with open(path, "w") as f:
        yaml.dump(cfg, f, sort_keys=False, allow_unicode=True)

write_task_yaml("coco_det.yaml", DET_ROOT)
write_task_yaml("coco_seg.yaml", SEG_ROOT)

print("coco_det.yaml y coco_seg.yaml generados.")


coco_det.yaml y coco_seg.yaml generados.


## 6. Modelo de detección YOLO preentrenado

Cargamos `yolov8s.pt`, un modelo YOLOv8 **small** ya preentrenado en el dataset COCO completo (80 clases). Elegimos la variante *small* como punto medio entre velocidad y precisión, apropiado para una GPU de 8GB (RTX 4060).


In [8]:
from ultralytics import YOLO

model_det = YOLO("yolov8s.pt")  # pesos preentrenados en COCO (transfer learning)
print(model_det.info())


YOLOv8s summary: 129 layers, 11,166,560 parameters, 0 gradients, 28.8 GFLOPs
(129, 11166560, 0, 28.8119296)


## 7. Entrenamiento del modelo de detección (transfer learning)

En lugar de entrenar desde cero, partimos de los pesos preentrenados (`yolov8s.pt`) y hacemos *fine-tuning* sobre nuestra partición de `coco_det.yaml`. Al arrancar desde pesos ya entrenados en COCO, el modelo converge en pocas épocas.

Parámetros elegidos para una RTX 4060 (8GB VRAM):
- `imgsz=640` (resolución estándar de YOLO)
- `batch=16` (cabe cómodamente en 8GB con yolov8s a 640px; bajar a 8 si hay error de memoria)
- `epochs=30` (suficiente para fine-tuning sobre un subconjunto pequeño; subir si se usa un dataset más grande)


In [9]:
results_det = model_det.train(
    data="coco_det.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project="runs_lab",
    name="coco_det_transfer",
    pretrained=True,   # asegura que se use transfer learning, no entrenamiento desde cero
    exist_ok=True,
)


Ultralytics 8.4.124  Python-3.11.13 torch-2.12.1+cpu CPU (Intel Core i7-9750H 2.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco_det.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=coco_det_transfer, nbs=64, nms=False, opset

## 8. Modelo de segmentación YOLO preentrenado

Análogamente, cargamos `yolov8s-seg.pt`, la variante de YOLOv8 preentrenada para **segmentación de instancias** sobre COCO.


**Limpieza de memoria GPU** antes de cargar el siguiente modelo (evita acumulación de VRAM entre entrenamientos sucesivos).

In [10]:
import gc

# Liberamos memoria GPU/CPU acumulada de la etapa anterior antes de cargar
# el siguiente modelo. Evita el error 'CUDA out of memory' cuando se corren
# varios entrenamientos seguidos en la misma sesión de kernel.
torch.cuda.empty_cache()
gc.collect()
if torch.cuda.is_available():
    print(f"VRAM libre: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB de "
          f"{torch.cuda.mem_get_info()[1] / 1e9:.2f} GB totales")


In [11]:
model_seg = YOLO("yolov8s-seg.pt")  # pesos preentrenados en COCO (transfer learning)
print(model_seg.info())


YOLOv8s-seg summary: 151 layers, 11,821,056 parameters, 0 gradients, 40.3 GFLOPs
(151, 11821056, 0, 40.3390976)


## 9. Entrenamiento del modelo de segmentación (transfer learning)

Usamos la misma partición 70/15/15 (mismas imágenes) pero apuntando al archivo `coco_seg.yaml`, cuyas etiquetas contienen los polígonos de segmentación.


In [12]:
results_seg = model_seg.train(
    data="coco_seg.yaml",
    epochs=30,
    imgsz=640,
    batch=16,
    device=DEVICE,
    project="runs_lab",
    name="coco_seg_transfer",
    pretrained=True,
    exist_ok=True,
)


Ultralytics 8.4.124  Python-3.11.13 torch-2.12.1+cpu CPU (Intel Core i7-9750H 2.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco_seg.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=coco_seg_transfer, nbs=64, nms=False, o

## 10. Evaluación del modelo de detección (sobre el conjunto de test)

### Ecuaciones de las métricas

**Precision** — de todo lo que el modelo predijo como positivo, ¿qué fracción era correcta?
$$Precision = \frac{TP}{TP + FP}$$

**Recall** — de todos los objetos reales, ¿qué fracción detectó el modelo?
$$Recall = \frac{TP}{TP + FN}$$

**F1-score** — media armónica entre precision y recall:
$$F1 = 2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$$

**Intersection over Union (IoU)** — mide qué tanto se traslapan la caja/máscara predicha ($A$) y la real ($B$):
$$IoU = \frac{|A \cap B|}{|A \cup B|}$$
Una predicción se cuenta como *True Positive* (TP) si su IoU respecto al ground-truth supera un umbral (por ejemplo 0.50).

**mAP@0.50** — mAP (mean Average Precision) calculado usando un único umbral de IoU = 0.50 para decidir aciertos, promediado sobre todas las clases.

**mAP@0.50:0.95** — el mismo cálculo pero promediado sobre 10 umbrales de IoU (0.50, 0.55, ..., 0.95), lo que da una medida más estricta y completa de la calidad de localización.

Donde $TP$ = verdaderos positivos, $FP$ = falsos positivos, $FN$ = falsos negativos.


In [13]:
metrics_det = model_det.val(data="coco_det.yaml", split="test", device=DEVICE)

precision_det = metrics_det.box.mp        # Precision promedio (todas las clases)
recall_det = metrics_det.box.mr           # Recall promedio
map50_det = metrics_det.box.map50         # mAP@0.50
map5095_det = metrics_det.box.map         # mAP@0.50:0.95
f1_det = (2 * precision_det * recall_det / (precision_det + recall_det)
          if (precision_det + recall_det) > 0 else 0.0)

print("=== Métricas de DETECCIÓN (test set) ===")
print(f"Precision      : {precision_det:.4f}")
print(f"Recall         : {recall_det:.4f}")
print(f"F1-score       : {f1_det:.4f}")
print(f"mAP@0.50       : {map50_det:.4f}")
print(f"mAP@0.50:0.95  : {map5095_det:.4f}")


Ultralytics 8.4.124  Python-3.11.13 torch-2.12.1+cpu CPU (Intel Core i7-9750H 2.60GHz)
Model summary (fused): 73 layers, 11,156,544 parameters, 0 gradients, 28.6 GFLOPs
WARNING val: Slow image access detected (ping: 0.30.1 ms, read: 4.11.3 MB/s, size: 43.6 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning C:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\dataset_coco\split\labels\test... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20 301.3it/s 0.1s
val: New cache created: C:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\dataset_coco\split\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.9s/it 5.8s<15.7s
                   all         20        106       0.74      0.775      0.873      0.704
                person          9         27

## 11. Evaluación del modelo de segmentación (sobre el conjunto de test)

### Coeficiente Dice

El coeficiente **Dice** (también llamado F1 a nivel de píxel) mide la similitud entre la máscara predicha ($A$) y la máscara real ($B$):

$$Dice = \frac{2 \, |A \cap B|}{|A| + |B|} = \frac{2\,TP}{2\,TP + FP + FN}$$

Se relaciona con IoU mediante: $Dice = \dfrac{2 \cdot IoU}{1 + IoU}$

Las demás métricas (Mask Precision, Mask Recall, Mask mAP@0.50, Mask mAP@0.50:0.95, IoU) se calculan igual que en detección, pero comparando **máscaras a nivel de píxel** en vez de bounding boxes.


In [14]:
metrics_seg = model_seg.val(data="coco_seg.yaml", split="test", device=DEVICE)

mask_precision = metrics_seg.seg.mp
mask_recall = metrics_seg.seg.mr
mask_map50 = metrics_seg.seg.map50
mask_map5095 = metrics_seg.seg.map

print("=== Métricas de SEGMENTACIÓN — máscaras (test set) ===")
print(f"Mask Precision     : {mask_precision:.4f}")
print(f"Mask Recall        : {mask_recall:.4f}")
print(f"Mask mAP@0.50      : {mask_map50:.4f}")
print(f"Mask mAP@0.50:0.95 : {mask_map5095:.4f}")


Ultralytics 8.4.124  Python-3.11.13 torch-2.12.1+cpu CPU (Intel Core i7-9750H 2.60GHz)
YOLOv8s-seg summary (fused): 86 layers, 11,810,560 parameters, 0 gradients, 40.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 209.070.0 MB/s, size: 43.5 KB)
val: Scanning C:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\dataset_coco\split\labels\test.cache... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.8s/it 9.6s<24.2s
                   all         20        106      0.741      0.768      0.849      0.693      0.741      0.768      0.846      0.609
                person          9         27          1      0.687      0.897      0.627          1      0.687      0.892      0.461
               bicycle          1          2      0.969          1      0.995      0.596      0.969         

In [15]:
# IoU y Dice promedio calculados directamente sobre las predicciones del test set,
# comparando cada máscara predicha contra su máscara ground-truth correspondiente.
import numpy as np
import cv2

def mask_iou_dice(pred_mask: np.ndarray, gt_mask: np.ndarray):
    """pred_mask y gt_mask son arreglos binarios (0/1) del mismo tamaño."""
    intersection = np.logical_and(pred_mask, gt_mask).sum()
    union = np.logical_or(pred_mask, gt_mask).sum()
    iou = intersection / union if union > 0 else 0.0
    dice = (2 * intersection) / (pred_mask.sum() + gt_mask.sum()) \
        if (pred_mask.sum() + gt_mask.sum()) > 0 else 0.0
    return iou, dice


test_images_seg = sorted((SEG_ROOT / "images" / "test").glob("*.jpg"))
iou_scores, dice_scores = [], []

for img_path in test_images_seg:
    result = model_seg.predict(source=str(img_path), device=DEVICE, verbose=False)[0]
    label_path = SEG_ROOT / "labels" / "test" / (img_path.stem + ".txt")
    if result.masks is None or not label_path.exists():
        continue

    h, w = result.orig_shape
    # Máscara ground-truth: reconstruimos el polígono normalizado del archivo .txt
    gt_mask = np.zeros((h, w), dtype=np.uint8)
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            coords = list(map(float, parts[1:]))
            pts = np.array(coords, dtype=np.float32).reshape(-1, 2)
            pts[:, 0] *= w
            pts[:, 1] *= h
            cv2.fillPoly(gt_mask, [pts.astype(np.int32)], 1)

    # Combinamos todas las máscaras predichas en una sola máscara binaria
    pred_mask = result.masks.data.cpu().numpy().sum(axis=0)
    pred_mask = (pred_mask > 0).astype(np.uint8)
    pred_mask = cv2.resize(pred_mask, (w, h), interpolation=cv2.INTER_NEAREST)

    iou, dice = mask_iou_dice(pred_mask, gt_mask)
    iou_scores.append(iou)
    dice_scores.append(dice)

mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
mean_dice = float(np.mean(dice_scores)) if dice_scores else 0.0

print(f"IoU promedio (test)  : {mean_iou:.4f}")
print(f"Dice promedio (test) : {mean_dice:.4f}")


IoU promedio (test)  : 0.8013
Dice promedio (test) : 0.8845


### Resumen de todas las métricas requeridas

In [16]:
import pandas as pd

summary = pd.DataFrame({
    "Métrica": [
        "Precision (det)", "Recall (det)", "F1-score (det)",
        "mAP@0.50 (det)", "mAP@0.50:0.95 (det)",
        "Mask Precision", "Mask Recall",
        "Mask mAP@0.50", "Mask mAP@0.50:0.95",
        "IoU", "Dice",
    ],
    "Valor": [
        precision_det, recall_det, f1_det, map50_det, map5095_det,
        mask_precision, mask_recall, mask_map50, mask_map5095,
        mean_iou, mean_dice,
    ],
})
summary["Valor"] = summary["Valor"].round(4)
summary


,Métrica,Valor
0,Precision (det),0.7400
1,Recall (det),0.7754
2,F1-score (det),0.7573
3,mAP@0.50 (det),0.8732
4,mAP@0.50:0.95 (det),0.7036
5,Mask Precision,0.7413
6,Mask Recall,0.7675
7,Mask mAP@0.50,0.8458
8,Mask mAP@0.50:0.95,0.6088
9,IoU,0.8013


## 12. Visualización de resultados

Para varias imágenes del conjunto de test mostramos, en una sola figura:

1. Imagen original con anotaciones **ground-truth** (caja + máscara real)
2. Predicción del modelo de **detección** (cajas + score de confianza)
3. Predicción del modelo de **segmentación** (máscaras + score de confianza)


In [17]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def draw_gt(ax, img_path, seg_root):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    ax.imshow(img)
    label_path = seg_root / "labels" / "test" / (img_path.stem + ".txt")
    if label_path.exists():
        with open(label_path) as f:
            for line in f:
                parts = line.strip().split()
                cls_id = int(parts[0])
                coords = list(map(float, parts[1:]))
                pts = np.array(coords).reshape(-1, 2)
                pts[:, 0] *= w
                pts[:, 1] *= h
                poly = patches.Polygon(pts, closed=True, fill=False,
                                        edgecolor="lime", linewidth=2)
                ax.add_patch(poly)
                ax.text(pts[:, 0].min(), pts[:, 1].min() - 4,
                        COCO_CLASSES[cls_id], color="lime", fontsize=8,
                        backgroundcolor="black")
    ax.set_title("Ground truth")
    ax.axis("off")


def draw_detection(ax, img_path):
    result = model_det.predict(source=str(img_path), device=DEVICE, verbose=False)[0]
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0])
        cls_id = int(box.cls[0])
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                  fill=False, edgecolor="red", linewidth=2)
        ax.add_patch(rect)
        ax.text(x1, y1 - 4, f"{COCO_CLASSES[cls_id]} {conf:.2f}",
                color="red", fontsize=8, backgroundcolor="black")
    ax.set_title("Detección (predicción)")
    ax.axis("off")


def draw_segmentation(ax, img_path):
    result = model_seg.predict(source=str(img_path), device=DEVICE, verbose=False)[0]
    ax.imshow(result.plot(boxes=False)[:, :, ::-1])  # plot() dibuja máscaras + conf
    ax.set_title("Segmentación (predicción)")
    ax.axis("off")


N_EXAMPLES = 4
sample_imgs = test_images_seg[:N_EXAMPLES]

fig, axes = plt.subplots(N_EXAMPLES, 3, figsize=(15, 5 * N_EXAMPLES))
for i, img_path in enumerate(sample_imgs):
    draw_gt(axes[i, 0], img_path, SEG_ROOT)
    draw_detection(axes[i, 1], img_path)
    draw_segmentation(axes[i, 2], img_path)

plt.tight_layout()
plt.savefig("resultados_visualizacion.png", dpi=150, bbox_inches="tight")
plt.show()


<Figure size 1500x2000 with 12 Axes>

## 13. Conclusiones

- Se aplicó **transfer learning** partiendo de los pesos `yolov8s.pt` / `yolov8s-seg.pt` (preentrenados en COCO), en vez de entrenar desde cero, tal como se solicitó.
- El dataset se particionó en 70% train / 15% val / 15% test de forma reproducible (semilla fija).
- Las anotaciones de detección se derivaron automáticamente de las de segmentación, garantizando consistencia entre ambas tareas sobre el mismo conjunto de objetos.
- Se reportaron todas las métricas solicitadas: Precision, Recall, F1, mAP@0.50, mAP@0.50:0.95 (detección) y Mask Precision, Mask Recall, Mask mAP@0.50, Mask mAP@0.50:0.95, IoU, Dice (segmentación).
- La visualización final compara, imagen por imagen, el ground-truth contra las predicciones de ambos modelos, incluyendo los scores de confianza.


---

## 14. Extensión opcional — escalando el laboratorio (dataset completo + modelo más grande)

Todo lo anterior (secciones 1-13) cumple los requisitos mínimos del laboratorio usando `coco128-seg` (128 imágenes) y `yolov8s`. Esta sección es una **extensión adicional**: se repite el mismo pipeline de transfer learning, pero escalado a:

- **Dataset:** COCO val2017 completo (**5,000 imágenes reales**, con las 80 clases de COCO bien representadas — no solo 1-2 instancias por clase como en el subconjunto de 128).
- **Modelo:** `yolov8m` / `yolov8m-seg` (medium, ~26M parámetros, vs. los ~11M de `yolov8s`).
- **Épocas:** 80, con `patience=15` (se detiene automáticamente si no mejora en 15 épocas seguidas, para no desperdiciar tiempo de GPU).

Se reutilizan las mismas funciones ya definidas arriba (`polygon_to_bbox`, `build_detection_labels`, `make_task_root`, `write_task_yaml`), aplicadas ahora sobre el dataset completo.

> ⚠️ Este entrenamiento es considerablemente más pesado. Con una RTX 4060 (8GB) y 16 núcleos de CPU, se estima del orden de 1 a 3 horas en total para ambos modelos — variará según tu equipo. Puedes correr esta sección de forma independiente (no depende de haber corrido nada más que la sección 1 de instalación y las definiciones de funciones de la sección 4-5).


### 14.1 Descarga de COCO val2017 (imágenes + anotaciones oficiales)

Descargamos el split de validación completo de COCO 2017: 5,000 imágenes con anotaciones completas de detección y segmentación (formato JSON original de COCO, no el formato YOLO que usa `coco128-seg`).


In [18]:
import shutil
from pathlib import Path
from ultralytics.utils.downloads import download

COCO_FULL_ROOT = Path("./dataset_coco_full")
COCO_FULL_ROOT.mkdir(exist_ok=True)

# Imágenes (≈1 GB, 5,000 imágenes)
download("http://images.cocodataset.org/zips/val2017.zip", dir=COCO_FULL_ROOT, unzip=True, delete=True)

# Anotaciones oficiales (≈241 MB; el zip trae train+val, solo usaremos instances_val2017.json)
download(
    "http://images.cocodataset.org/annotations/annotations_trainval2017.zip",
    dir=COCO_FULL_ROOT, unzip=True, delete=True,
)

FULL_IMAGES_DIR = COCO_FULL_ROOT / "val2017"
print(f"Imágenes descargadas: {len(list(FULL_IMAGES_DIR.glob('*.jpg')))}")


Unzipping dataset_coco_full\val2017.zip to C:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\dataset_coco_full\val2017...: 100% ━━━━━━━━━━━━ 5001/5001 450.5files/s 11.1s0.0s
Unzipping dataset_coco_full\annotations_trainval2017.zip to C:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\dataset_coco_full\annotations...: 100% ━━━━━━━━━━━━ 6/6 1.3files/s 4.5s0.7ss
Imágenes descargadas: 5000


### 14.2 Conversión de anotaciones: COCO JSON → formato YOLO

Usamos el conversor oficial de Ultralytics, que mapea automáticamente los 91 IDs de categoría originales de COCO a los 80 IDs que usan los pesos preentrenados (`cls91to80=True`), y extrae tanto los polígonos de segmentación como los bounding boxes.


In [19]:
from ultralytics.data.converter import convert_coco

# Aislamos solo el archivo de anotaciones de validación (evita procesar
# también instances_train2017.json, que son ~118,000 imágenes que no usaremos)
VAL_ANN_DIR = COCO_FULL_ROOT / "val_annotations_only"
VAL_ANN_DIR.mkdir(exist_ok=True)
shutil.copy(
    COCO_FULL_ROOT / "annotations" / "instances_val2017.json",
    VAL_ANN_DIR / "instances_val2017.json",
)

convert_coco(
    labels_dir=str(VAL_ANN_DIR),
    save_dir=str(COCO_FULL_ROOT / "yolo_labels"),
    use_segments=True,     # incluye polígonos de segmentación
    use_keypoints=False,
    cls91to80=True,        # mapea a los mismos 80 IDs que usan los pesos preentrenados
)

FULL_LABELS_DIR = COCO_FULL_ROOT / "yolo_labels" / "labels" / "val2017"
print(f"Labels generados: {len(list(FULL_LABELS_DIR.glob('*.txt')))}")


Annotations C:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\dataset_coco_full\val_annotations_only\instances_val2017.json: 100% ━━━━━━━━━━━━ 4952/4952 615.5it/s 8.0s0.1s
COCO data converted successfully.
Results saved to C:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\dataset_coco_full\yolo_labels
Labels generados: 4952


### 14.3 Partición 70/15/15 y generación de labels de detección

Mismo procedimiento que en la sección 3-4, aplicado ahora a las 5,000 imágenes.


In [20]:
random.seed(42)

FULL_SPLIT_ROOT = Path("./dataset_coco_full/split")
if FULL_SPLIT_ROOT.exists():
    shutil.rmtree(FULL_SPLIT_ROOT)

all_images_full = sorted(FULL_IMAGES_DIR.glob("*.jpg"))
random.shuffle(all_images_full)

n_total_full = len(all_images_full)
n_train_full = int(n_total_full * 0.70)
n_val_full = int(n_total_full * 0.15)
splits_full = {
    "train": all_images_full[:n_train_full],
    "val": all_images_full[n_train_full:n_train_full + n_val_full],
    "test": all_images_full[n_train_full + n_val_full:],
}

for split_name, img_list in splits_full.items():
    img_out = FULL_SPLIT_ROOT / "images" / split_name
    lbl_out = FULL_SPLIT_ROOT / "labels" / split_name
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)
    for img_path in img_list:
        label_path = FULL_LABELS_DIR / (img_path.stem + ".txt")
        shutil.copy(img_path, img_out / img_path.name)
        if label_path.exists():
            shutil.copy(label_path, lbl_out / label_path.name)

for split_name, img_list in splits_full.items():
    pct = len(img_list) / n_total_full * 100
    print(f"  {split_name:5s}: {len(img_list):5d} imágenes ({pct:.1f}%)")

# Generamos labels de detección (bbox) a partir de los de segmentación,
# reutilizando la misma función definida en la sección 4
for split_name in ["train", "val", "test"]:
    seg_dir = FULL_SPLIT_ROOT / "labels" / split_name
    det_dir = FULL_SPLIT_ROOT / "labels_det" / split_name
    build_detection_labels(seg_dir, det_dir)


  train:  3500 imágenes (70.0%)
  val  :   750 imágenes (15.0%)
  test :   750 imágenes (15.0%)


### 14.4 Archivos de configuración (data.yaml)

In [21]:
def make_task_root_full(task_labels_dirname, task_root_name):
    task_root = Path(f"./dataset_coco_full/{task_root_name}")
    for split_name in ["train", "val", "test"]:
        img_src = (FULL_SPLIT_ROOT / "images" / split_name).resolve()
        lbl_src = (FULL_SPLIT_ROOT / task_labels_dirname / split_name).resolve()
        img_dst = task_root / "images" / split_name
        lbl_dst = task_root / "labels" / split_name
        img_dst.parent.mkdir(parents=True, exist_ok=True)
        lbl_dst.parent.mkdir(parents=True, exist_ok=True)
        if img_dst.exists() or img_dst.is_symlink():
            img_dst.unlink()
        if lbl_dst.exists() or lbl_dst.is_symlink():
            lbl_dst.unlink()
        img_dst.symlink_to(img_src)
        lbl_dst.symlink_to(lbl_src)
    return task_root

DET_ROOT_FULL = make_task_root_full("labels_det", "det_task_full")
SEG_ROOT_FULL = make_task_root_full("labels", "seg_task_full")

write_task_yaml("coco_det_full.yaml", DET_ROOT_FULL)
write_task_yaml("coco_seg_full.yaml", SEG_ROOT_FULL)
print("coco_det_full.yaml y coco_seg_full.yaml generados.")


coco_det_full.yaml y coco_seg_full.yaml generados.


### 14.5 Transfer learning con `yolov8m` — detección (80 épocas)

`workers=12` para aprovechar tus 16 núcleos de CPU en la carga de datos. `patience=15` detiene el entrenamiento automáticamente si no hay mejora en 15 épocas.


**Limpieza de memoria GPU** antes de cargar el siguiente modelo (evita acumulación de VRAM entre entrenamientos sucesivos).

In [22]:
import gc

# Liberamos memoria GPU/CPU acumulada de la etapa anterior antes de cargar
# el siguiente modelo. Evita el error 'CUDA out of memory' cuando se corren
# varios entrenamientos seguidos en la misma sesión de kernel.
torch.cuda.empty_cache()
gc.collect()
if torch.cuda.is_available():
    print(f"VRAM libre: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB de "
          f"{torch.cuda.mem_get_info()[1] / 1e9:.2f} GB totales")


In [23]:
model_det_full = YOLO("yolov8m.pt")

results_det_full = model_det_full.train(
    data="coco_det_full.yaml",
    epochs=80,
    imgsz=640,
    batch=8,
    device=DEVICE,
    workers=0,
    patience=40,
    project="runs_lab_full",
    name="coco_det_transfer_full",
    pretrained=True,
    exist_ok=True,
)


Ultralytics 8.4.124  Python-3.11.13 torch-2.12.1+cpu CPU (Intel Core i7-9750H 2.60GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco_det_full.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=coco_det_transfer_full, nbs=64, nms=Fal

KeyboardInterrupt: 

### 14.6 Transfer learning con `yolov8m-seg` — segmentación (80 épocas)

**Limpieza de memoria GPU** antes de cargar el siguiente modelo (evita acumulación de VRAM entre entrenamientos sucesivos).

In [24]:
import gc

# Liberamos memoria GPU/CPU acumulada de la etapa anterior antes de cargar
# el siguiente modelo. Evita el error 'CUDA out of memory' cuando se corren
# varios entrenamientos seguidos en la misma sesión de kernel.
torch.cuda.empty_cache()
gc.collect()
if torch.cuda.is_available():
    print(f"VRAM libre: {torch.cuda.mem_get_info()[0] / 1e9:.2f} GB de "
          f"{torch.cuda.mem_get_info()[1] / 1e9:.2f} GB totales")


In [5]:
from pathlib import Path
import pandas as pd

det_run = Path(
    "runs/detect/runs_lab_full/coco_det_transfer_full"
)

results_path = det_run / "results.csv"
last_path = det_run / "weights" / "last.pt"
best_path = det_run / "weights" / "best.pt"

history = pd.read_csv(results_path)

# Eliminar espacios que Ultralytics puede colocar en los encabezados
history.columns = history.columns.str.strip()

print("Épocas registradas:", len(history))
print("Última época registrada:", history["epoch"].iloc[-1])

print("\nCheckpoint para continuar:")
print(last_path.resolve())

print("\nCheckpoint con el mejor resultado:")
print(best_path.resolve())

print("\nÚltimas filas del entrenamiento:")
display(history.tail())

Épocas registradas: 3
Última época registrada: 3

Checkpoint para continuar:
C:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\runs\detect\runs_lab_full\coco_det_transfer_full\weights\last.pt

Checkpoint con el mejor resultado:
C:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\runs\detect\runs_lab_full\coco_det_transfer_full\weights\best.pt

Últimas filas del entrenamiento:


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
0,1,8598.48,0.93915,1.11482,1.09972,0.66806,0.62022,0.65060,0.48555,0.94768,0.97410,1.08474,0.000040,0.000040,0.000040
1,2,17262.40,0.95265,1.14641,1.11493,0.66997,0.56106,0.60882,0.44470,0.99549,1.08439,1.13731,0.000078,0.000078,0.000078
2,3,25710.70,0.96591,1.17987,1.12533,0.60576,0.55917,0.57601,0.42165,1.01770,1.16902,1.14273,0.000116,0.000116,0.000116


In [ ]:
""""
import gc
import torch
from ultralytics import YOLO

# Eliminar el checkpoint cargado en RAM
if "checkpoint" in globals():
    del checkpoint

gc.collect()
torch.cuda.empty_cache()

model_det_full = YOLO(str(cuda_checkpoint))

results_det_full = model_det_full.train(
    resume=True,
    device=0,
    batch=1,
    workers=2,
    amp=True,
    deterministic=False,
) """"

### 14.7 Evaluación en test (dataset completo)

In [1]:
""""
model_seg_full = YOLO("yolov8m-seg.pt")

results_seg_full = model_seg_full.train(
    data="coco_seg_full.yaml",
    epochs=80,
    imgsz=640,
    batch=8,
    device=DEVICE,
    workers=0,
    patience=40,
    project="runs_lab_full",
    name="coco_seg_transfer_full",
    pretrained=True,
    exist_ok=True,
)
"""""

import torch
from ultralytics import YOLO

DEVICE = 0

model_det_quick = YOLO("yolov8n.pt")

results_det_quick = model_det_quick.train(
    data="coco_det_full.yaml",

    epochs=3,
    imgsz=320,
    batch=4,

    device=DEVICE,
    workers=2,
    amp=True,
    deterministic=False,

    patience=2,
    val=True,
    plots=True,

    project="runs_lab_quick",
    name="coco_det_quick",
    exist_ok=True,
)


Ultralytics 8.4.124  Python-3.11.13 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1050, 3072MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco_det_full.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=coco_det_quick, nbs=64, nms=Fals

In [2]:
import gc
import torch
from ultralytics import YOLO

gc.collect()
torch.cuda.empty_cache()

model_seg_quick = YOLO("yolov8n-seg.pt")

results_seg_quick = model_seg_quick.train(
    data="coco_seg_full.yaml",
    epochs=3,
    imgsz=320,
    batch=4,
    device=0,
    workers=2,
    amp=True,
    deterministic=False,
    patience=2,
    val=True,
    plots=True,
    project="runs_lab_quick",
    name="coco_seg_quick",
    exist_ok=True,
)

Ultralytics 8.4.124  Python-3.11.13 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1050, 3072MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=coco_seg_full.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=3, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=coco_seg_quick, nbs=64, nms=

In [5]:
"""""
metrics_det_full = model_det_full.val(data="coco_det_full.yaml", split="test", device=DEVICE)
metrics_seg_full = model_seg_full.val(data="coco_seg_full.yaml", split="test", device=DEVICE)

f1_det_full = (
    2 * metrics_det_full.box.mp * metrics_det_full.box.mr
    / (metrics_det_full.box.mp + metrics_det_full.box.mr)
    if (metrics_det_full.box.mp + metrics_det_full.box.mr) > 0 else 0.0
)

summary_full = pd.DataFrame({
    "Métrica": [
        "Precision (det)", "Recall (det)", "F1-score (det)",
        "mAP@0.50 (det)", "mAP@0.50:0.95 (det)",
        "Mask Precision", "Mask Recall",
        "Mask mAP@0.50", "Mask mAP@0.50:0.95",
    ],
    "Valor (128 imgs, yolov8s)": [
        precision_det, recall_det, f1_det, map50_det, map5095_det,
        mask_precision, mask_recall, mask_map50, mask_map5095,
    ],
    "Valor (5000 imgs, yolov8m)": [
        metrics_det_full.box.mp, metrics_det_full.box.mr, f1_det_full,
        metrics_det_full.box.map50, metrics_det_full.box.map,
        metrics_seg_full.seg.mp, metrics_seg_full.seg.mr,
        metrics_seg_full.seg.map50, metrics_seg_full.seg.map,
    ],
})
summary_full.iloc[:, 1:] = summary_full.iloc[:, 1:].round(4)
summary_full
"""""

import pandas as pd
from pathlib import Path
from ultralytics import YOLO

DEVICE = 0

# ============================================================
# 1. EXPERIMENTO DE 128 IMÁGENES CON YOLOv8s
# ============================================================

det_128_path = Path(
    "runs/detect/runs_lab/"
    "coco_det_transfer/weights/best.pt"
)

seg_128_path = Path(
    "runs/segment/runs_lab/"
    "coco_seg_transfer/weights/best.pt"
)

model_det_128 = YOLO(str(det_128_path))
model_seg_128 = YOLO(str(seg_128_path))

metrics_det_128 = model_det_128.val(
    data="coco_det.yaml",
    split="test",
    imgsz=640,
    batch=4,
    device=DEVICE,
    workers=2,
)

metrics_seg_128 = model_seg_128.val(
    data="coco_seg.yaml",
    split="test",
    imgsz=640,
    batch=4,
    device=DEVICE,
    workers=2,
)

# Variables que solicitaba la celda original
precision_det = metrics_det_128.box.mp
recall_det = metrics_det_128.box.mr

f1_det = (
    2 * precision_det * recall_det
    / (precision_det + recall_det)
    if (precision_det + recall_det) > 0
    else 0.0
)

map50_det = metrics_det_128.box.map50
map5095_det = metrics_det_128.box.map

mask_precision = metrics_seg_128.seg.mp
mask_recall = metrics_seg_128.seg.mr
mask_map50 = metrics_seg_128.seg.map50
mask_map5095 = metrics_seg_128.seg.map


# ============================================================
# 2. RESULTADOS RÁPIDOS DE 5000 IMÁGENES
# ============================================================

# Solo repite la evaluación si las variables se perdieron
if "metrics_det_quick" not in globals():
    model_det_quick = YOLO(
        "runs/detect/runs_lab_quick/"
        "coco_det_quick/weights/best.pt"
    )

    metrics_det_quick = model_det_quick.val(
        data="coco_det_full.yaml",
        split="test",
        imgsz=320,
        batch=4,
        device=DEVICE,
        workers=2,
    )

if "metrics_seg_quick" not in globals():
    model_seg_quick = YOLO(
        "runs/segment/runs_lab_quick/"
        "coco_seg_quick/weights/best.pt"
    )

    metrics_seg_quick = model_seg_quick.val(
        data="coco_seg_full.yaml",
        split="test",
        imgsz=320,
        batch=4,
        device=DEVICE,
        workers=2,
    )

precision_quick = metrics_det_quick.box.mp
recall_quick = metrics_det_quick.box.mr

f1_det_quick = (
    2 * precision_quick * recall_quick
    / (precision_quick + recall_quick)
    if (precision_quick + recall_quick) > 0
    else 0.0
)


# ============================================================
# 3. TABLA COMPARATIVA
# ============================================================

summary_full = pd.DataFrame({
    "Métrica": [
        "Precision (det)",
        "Recall (det)",
        "F1-score (det)",
        "mAP@0.50 (det)",
        "mAP@0.50:0.95 (det)",
        "Mask Precision",
        "Mask Recall",
        "Mask mAP@0.50",
        "Mask mAP@0.50:0.95",
    ],

    "Valor (128 imgs, yolov8s, 640px)": [
        precision_det,
        recall_det,
        f1_det,
        map50_det,
        map5095_det,
        mask_precision,
        mask_recall,
        mask_map50,
        mask_map5095,
    ],

    "Valor (5000 imgs, yolov8n, 320px, 3 épocas)": [
        metrics_det_quick.box.mp,
        metrics_det_quick.box.mr,
        f1_det_quick,
        metrics_det_quick.box.map50,
        metrics_det_quick.box.map,
        metrics_seg_quick.seg.mp,
        metrics_seg_quick.seg.mr,
        metrics_seg_quick.seg.map50,
        metrics_seg_quick.seg.map,
    ],
})

summary_full.iloc[:, 1:] = (
    summary_full.iloc[:, 1:]
    .astype(float)
    .round(4)
)

summary_full

Ultralytics 8.4.124  Python-3.11.13 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce GTX 1050, 3072MiB)
Model summary (fused): 73 layers, 11,156,544 parameters, 0 gradients, 28.6 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 111.315.0 MB/s, size: 49.6 KB)
val: Scanning C:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\dataset_coco\split\labels\test.cache... 20 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20/20  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.2it/s 4.1s0.3s6s
                   all         20        106      0.709      0.753      0.847      0.674
                person          9         27      0.987      0.667      0.916      0.665
               bicycle          1          2      0.831        0.5      0.638      0.268
                   car          2          4      0.319       0.25      0.501      0.236
            motorcycle          1          2      0.814    

,Métrica,"Valor (128 imgs, yolov8s, 640px)","Valor (5000 imgs, yolov8n, 320px, 3 épocas)"
0,Precision (det),0.7086,0.5776
1,Recall (det),0.7525,0.3752
2,F1-score (det),0.7299,0.4549
3,mAP@0.50 (det),0.8471,0.4027
4,mAP@0.50:0.95 (det),0.6742,0.2715
5,Mask Precision,0.7034,0.5189
6,Mask Recall,0.7576,0.3539
7,Mask mAP@0.50,0.8327,0.3605
8,Mask mAP@0.50:0.95,0.5902,0.2163


### Resultado principal: pérdida de generalización durante el ajuste fino 😳💥
Los modelos de detección y segmentación se ajustaron a partir de los pesos preentrenados `yolov8m.pt` y `yolov8m-seg.pt`, utilizando 5,000 imágenes provenientes de COCO val2017. Inicialmente, el entrenamiento se configuró con `patience=15`. Posteriormente, se incrementó este valor a `patience=40` para comprobar si el entrenamiento se estaba deteniendo antes de alcanzar un mejor desempeño.

Sin embargo, ampliar la paciencia no produjo una mejora. En ambos modelos, el checkpoint con los mejores resultados (`best.pt`) correspondió a la primera época. A partir de ese momento, ninguna de las épocas posteriores logró superar el rendimiento inicial y el entrenamiento terminó después de 41 épocas debido al criterio de parada temprana.

El análisis de las curvas permite observar dos comportamientos opuestos. Por un lado, las funciones de pérdida calculadas sobre el conjunto de entrenamiento, como `box_loss`, `cls_loss` y `seg_loss`, disminuyeron progresivamente. En el modelo de detección, por ejemplo, `box_loss` pasó de 0.938 en la primera época a 0.771 en la época 41. Por otro lado, el desempeño sobre los datos de validación se deterioró: el mAP@0.50 disminuyó de 0.651 a 0.525 en detección y de 0.633 a 0.527 en segmentación.

Este patrón es característico del sobreajuste. Aunque los modelos aprendieron a representar cada vez mejor las imágenes utilizadas durante el entrenamiento, su capacidad para responder correctamente ante datos no utilizados en la actualización de los pesos se redujo.

Una posible explicación se encuentra en la relación entre los datos de preentrenamiento y el subconjunto empleado para el fine-tuning. Los pesos originales de YOLOv8 fueron obtenidos mediante el entrenamiento con aproximadamente 118,000 imágenes de COCO train2017, un conjunto considerablemente más amplio y diverso que las 3,500 imágenes destinadas al entrenamiento en esta extensión. Además, ambos conjuntos pertenecen a la misma distribución general de COCO.

Por esta razón, los modelos ya contaban desde el inicio con representaciones suficientemente generales para resolver la tarea. Al continuar modificando sus parámetros con un subconjunto mucho más reducido, comenzaron a adaptarse excesivamente a las características particulares de esa partición. Como consecuencia, se alejaron de la solución general aprendida durante el preentrenamiento y disminuyeron su capacidad de generalización. El mejor resultado apareció en la época 1 porque, en ese momento, los pesos todavía conservaban gran parte de la información adquirida originalmente.

Este resultado difiere del observado en el laboratorio base con `coco128-seg`, donde se utilizaron únicamente 89 imágenes para el entrenamiento. En ese caso, el ajuste fino produjo beneficios durante las primeras épocas. Una explicación probable es que dicho conjunto presentaba una composición de clases más particular, incluyendo categorías con una o dos instancias, por lo que el modelo sí necesitaba adaptarse a esas condiciones específicas. En cambio, el subconjunto de 3,500 imágenes de val2017 conserva suficientemente bien las características generales de COCO, haciendo menos necesario un ajuste adicional de todos los parámetros.

### Conclusión obtenida del experimento 👽🔥

Los resultados muestran que aumentar el número de imágenes o utilizar una arquitectura de mayor capacidad no conduce automáticamente a un mejor desempeño. La efectividad del fine-tuning depende de la cantidad y diversidad de los datos disponibles, así como de su semejanza con la distribución utilizada durante el preentrenamiento.

Cuando el conjunto de ajuste es considerablemente menor que el conjunto original y ambos contienen información similar, una actualización intensa de los pesos puede perjudicar el rendimiento previamente adquirido. Para evitar este comportamiento, sería conveniente aplicar una tasa de aprendizaje más pequeña, mantener congeladas más capas del modelo o limitar el número de épocas. También resulta importante evaluar cuidadosamente el desempeño desde las primeras épocas, utilizando el resultado inicial como referencia para detectar oportunamente cualquier pérdida de generalización.


In [4]:
from pathlib import Path
import yaml

search_root = Path.cwd()

checkpoints = sorted(search_root.rglob("best.pt"))

print(f"Checkpoints encontrados: {len(checkpoints)}\n")

for index, checkpoint_path in enumerate(checkpoints):
    run_dir = checkpoint_path.parent.parent
    args_path = run_dir / "args.yaml"

    print(f"[{index}] {checkpoint_path}")

    if args_path.exists():
        with open(args_path, "r", encoding="utf-8") as file:
            args = yaml.safe_load(file)

        print("    task:", args.get("task"))
        print("    model:", args.get("model"))
        print("    data:", args.get("data"))
        print("    imgsz:", args.get("imgsz"))
        print("    epochs:", args.get("epochs"))
        print("    name:", args.get("name"))

    print()

Checkpoints encontrados: 5

[0] c:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\runs\detect\runs_lab\coco_det_transfer\weights\best.pt
    task: detect
    model: yolov8s.pt
    data: coco_det.yaml
    imgsz: 640
    epochs: 30
    name: coco_det_transfer

[1] c:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\runs\detect\runs_lab_full\coco_det_transfer_full\weights\best.pt
    task: detect
    model: runs\detect\runs_lab_full\coco_det_transfer_full\weights\last_cuda.pt
    data: coco_det_full.yaml
    imgsz: 640
    epochs: 80
    name: coco_det_transfer_full

[2] c:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\runs\detect\runs_lab_quick\coco_det_quick\weights\best.pt
    task: detect
    model: yolov8n.pt
    data: coco_det_full.yaml
    imgsz: 320
    epochs: 3
    name: coco_det_quick

[3] c:\Specialization_Artificial_Intelligence_Cinvestav\Module4_MarioValdovinos\runs\segment\runs_lab\coco_seg_transfe